Preprocessing the GRN files

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

# ----------------------------
# CONFIG
# ----------------------------
BENCH_ROOT = Path("/Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/benchmarking/data/GRN")
OUT_DIR = BENCH_ROOT / "preprocessed"

RECURSIVE = False
DO_ZSCORE = True

# ----------------------------
# Helpers
# ----------------------------
def zscore_rows(X: pd.DataFrame) -> pd.DataFrame:
    mu = X.mean(axis=1)
    sd = X.std(axis=1, ddof=1).replace(0, np.nan)
    Z = X.sub(mu, axis=0).div(sd, axis=0)
    return Z.dropna(axis=0)

def preprocess_grn_csv(in_csv: Path, out_tsv: Path, do_zscore: bool = True):
    """
    GRNbenchmark expression format:
      - first column: gene IDs (often called 'Row')
      - remaining columns: samples/replicates
      - orientation already genes x samples

    Output:
      - TSV in genes x samples format for Piglasso
    """
    df = pd.read_csv(in_csv)

    if df.shape[1] < 2:
        raise ValueError(f"{in_csv.name} has fewer than 2 columns.")

    # Use first column as gene index, regardless of exact name
    gene_col = df.columns[0]
    df = df.set_index(gene_col)

    # Clean gene names
    df.index = df.index.astype(str).str.strip()

    # Keep only sample columns
    X = df.apply(pd.to_numeric, errors="coerce")

    print(f"   Raw shape (genes x samples): {X.shape}")

    # Drop genes with all missing values
    X = X.dropna(axis=0, how="all")

    # Fill remaining missing values with 0
    X = X.fillna(0.0)

    # Remove duplicated gene names if present: keep first
    if X.index.duplicated().any():
        n_dup = int(X.index.duplicated().sum())
        print(f"   [WARN] Found {n_dup} duplicated gene names; keeping first occurrence.")
        X = X[~X.index.duplicated(keep="first")]

    if do_zscore:
        X = zscore_rows(X)
        print(f"   Z-scored shape (genes x samples): {X.shape}")

    out_tsv.parent.mkdir(parents=True, exist_ok=True)
    out_tsv = out_tsv.with_suffix(".tsv")
    X.to_csv(out_tsv, sep="\t")

    return X.shape

# ----------------------------
# Find GRNbenchmark files
# ----------------------------
pattern = "*GeneExpression.csv" if not RECURSIVE else "**/*.csv"
files = sorted(BENCH_ROOT.glob(pattern))

# Avoid reprocessing gold-standard or already processed folders if needed
files = [
    f for f in files
    if f.is_file() and "preprocessed" not in f.parts
]

if not files:
    raise FileNotFoundError(f"No .csv files found in {BENCH_ROOT.resolve()}")

print(f"[INFO] Found {len(files)} GRNbenchmark expression files.")
for f in files:
    print("  -", f.name)

# ----------------------------
# Process all
# ----------------------------
summary = []

for i, fp in enumerate(files, start=1):
    print(f"\n[{i}/{len(files)}] Processing: {fp.name}")

    out_fp = OUT_DIR / fp.stem  # .tsv added inside function

    shape = preprocess_grn_csv(fp, out_fp, do_zscore=DO_ZSCORE)

    print(f"   -> wrote {out_fp.with_suffix('.tsv').name}  shape={shape}")

    summary.append({
        "input": fp.name,
        "output": out_fp.with_suffix(".tsv").name,
        "genes": shape[0],
        "samples": shape[1]
    })

summary_df = pd.DataFrame(summary)
display(summary_df)

[INFO] Found 30 GRNbenchmark expression files.
  - GeneNetWeaver_HighNoise_Network1_GeneExpression.csv
  - GeneNetWeaver_HighNoise_Network2_GeneExpression.csv
  - GeneNetWeaver_HighNoise_Network3_GeneExpression.csv
  - GeneNetWeaver_HighNoise_Network4_GeneExpression.csv
  - GeneNetWeaver_HighNoise_Network5_GeneExpression.csv
  - GeneNetWeaver_LowNoise_Network1_GeneExpression.csv
  - GeneNetWeaver_LowNoise_Network2_GeneExpression.csv
  - GeneNetWeaver_LowNoise_Network3_GeneExpression.csv
  - GeneNetWeaver_LowNoise_Network4_GeneExpression.csv
  - GeneNetWeaver_LowNoise_Network5_GeneExpression.csv
  - GeneNetWeaver_MediumNoise_Network1_GeneExpression.csv
  - GeneNetWeaver_MediumNoise_Network2_GeneExpression.csv
  - GeneNetWeaver_MediumNoise_Network3_GeneExpression.csv
  - GeneNetWeaver_MediumNoise_Network4_GeneExpression.csv
  - GeneNetWeaver_MediumNoise_Network5_GeneExpression.csv
  - GeneSPIDER_HighNoise_Network1_GeneExpression.csv
  - GeneSPIDER_HighNoise_Network2_GeneExpression.csv
  

,input,output,genes,samples
0,GeneNetWeaver_HighNoise_Network1_GeneExpressio...,GeneNetWeaver_HighNoise_Network1_GeneExpressio...,100,300
1,GeneNetWeaver_HighNoise_Network2_GeneExpressio...,GeneNetWeaver_HighNoise_Network2_GeneExpressio...,100,300
2,GeneNetWeaver_HighNoise_Network3_GeneExpressio...,GeneNetWeaver_HighNoise_Network3_GeneExpressio...,100,300
3,GeneNetWeaver_HighNoise_Network4_GeneExpressio...,GeneNetWeaver_HighNoise_Network4_GeneExpressio...,100,300
4,GeneNetWeaver_HighNoise_Network5_GeneExpressio...,GeneNetWeaver_HighNoise_Network5_GeneExpressio...,100,300
5,GeneNetWeaver_LowNoise_Network1_GeneExpression...,GeneNetWeaver_LowNoise_Network1_GeneExpression...,100,300
6,GeneNetWeaver_LowNoise_Network2_GeneExpression...,GeneNetWeaver_LowNoise_Network2_GeneExpression...,100,300
7,GeneNetWeaver_LowNoise_Network3_GeneExpression...,GeneNetWeaver_LowNoise_Network3_GeneExpression...,100,300
8,GeneNetWeaver_LowNoise_Network4_GeneExpression...,GeneNetWeaver_LowNoise_Network4_GeneExpression...,100,300
9,GeneNetWeaver_LowNoise_Network5_GeneExpression...,GeneNetWeaver_LowNoise_Network5_GeneExpression...,100,300


After running all these files through piglasso, we have to reformat the output so that we can submit it to GRNbenchmark

In [4]:
from pathlib import Path
import re
import pickle
import zipfile
import numpy as np
import pandas as pd
from typing import Optional
# ----------------------------
# CONFIG
# ----------------------------
PROJECT_ROOT = Path("/Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries")

IN_DIR = PROJECT_ROOT / "benchmarking" / "results" / "piglasso" / "GRN"
OUT_DIR = IN_DIR / "reformatted"
OUT_DIR.mkdir(parents=True, exist_ok=True)

PIG_SCORE_MODE = "max"   # "max" or "mean"

# ----------------------------
# Helpers
# ----------------------------
def load_piglasso_pkl(pkl_path: Path) -> dict:
    with open(pkl_path, "rb") as f:
        return pickle.load(f)

def piglasso_scores_from_payload(payload: dict, mode: str = "max") -> tuple[list[str], np.ndarray]:
    E = np.array(payload["edge_counts_all"], dtype=float)
    succ = np.array(payload["success_counts"], dtype=float)
    genes = [str(g).strip() for g in payload["genes"]]

    succ = np.where(succ <= 0, np.nan, succ)
    S = E / succ.reshape(1, 1, -1)

    if mode == "max":
        score_mat = np.nanmax(S, axis=2)
    elif mode == "mean":
        score_mat = np.nanmean(S, axis=2)
    else:
        raise ValueError("mode must be 'max' or 'mean'")

    np.fill_diagonal(score_mat, 0.0)
    score_mat = np.nan_to_num(score_mat, nan=0.0)

    return genes, score_mat

def undirected_to_bidirectional_submission(
    genes: list[str],
    score_mat: np.ndarray,
    out_csv: Path,
    sign_value: int = 1,
):
    rows = []
    p = len(genes)

    for i in range(p):
        for j in range(i + 1, p):
            g1, g2 = genes[i], genes[j]
            w = float(max(score_mat[i, j], score_mat[j, i]))

            if w <= 0:
                continue

            rows.append((g1, g2, w, sign_value))
            rows.append((g2, g1, w, sign_value))

    df = pd.DataFrame(rows, columns=["Regulator", "Target", "Weight", "Sign"])
    df = df.sort_values("Weight", ascending=False).reset_index(drop=True)
    df.to_csv(out_csv, index=False)
    return df

def parse_source_family(name: str) -> Optional[str]:
    if name.startswith("GeneNetWeaver"):
        return "GeneNetWeaver"
    if name.startswith("GeneSPIDER"):
        return "GeneSPIDER"
    return None

def parse_noise_level(name: str) -> Optional[str]:
    patterns = {
        "LowNoise": [r"LowNoise", r"low[_\-]?noise", r"\blow\b"],
        "MediumNoise": [r"MediumNoise", r"medium[_\-]?noise", r"\bmedium\b", r"\bmed\b"],
        "HighNoise": [r"HighNoise", r"high[_\-]?noise", r"\bhigh\b"],
    }
    for label, pats in patterns.items():
        for pat in pats:
            if re.search(pat, name, flags=re.IGNORECASE):
                return label
    return None

def parse_network_number(name: str) -> Optional[str]:
    patterns = [
        r"Network[_\-]?([1-5])",
        r"net[_\-]?([1-5])",
    ]
    for pat in patterns:
        m = re.search(pat, name, flags=re.IGNORECASE)
        if m:
            return f"Network{m.group(1)}"
    return None

def make_output_filename(pkl_name: str) -> str:
    family = parse_source_family(pkl_name)
    noise = parse_noise_level(pkl_name)
    network = parse_network_number(pkl_name)

    if family is None:
        raise ValueError(f"Could not detect source family from filename: {pkl_name}")
    if noise is None:
        raise ValueError(f"Could not detect noise level from filename: {pkl_name}")
    if network is None:
        raise ValueError(f"Could not detect network number from filename: {pkl_name}")

    return f"{family}_{noise}_{network}_grn.csv"

# ----------------------------
# Find all pkl files
# ----------------------------
pkl_files = sorted(IN_DIR.glob("*.pkl"))
if not pkl_files:
    raise FileNotFoundError(f"No .pkl files found in {IN_DIR}")

print(f"[INFO] Found {len(pkl_files)} Piglasso result files.")

summary = []

# ----------------------------
# Convert all
# ----------------------------
for i, pkl_path in enumerate(pkl_files, start=1):
    print(f"\n[{i}/{len(pkl_files)}] Processing: {pkl_path.name}")

    try:
        out_name = make_output_filename(pkl_path.name)
    except ValueError as e:
        print(f"[SKIP] {e}")
        continue

    out_csv = OUT_DIR / out_name

    payload = load_piglasso_pkl(pkl_path)
    genes, score_mat = piglasso_scores_from_payload(payload, mode=PIG_SCORE_MODE)

    df_out = undirected_to_bidirectional_submission(
        genes=genes,
        score_mat=score_mat,
        out_csv=out_csv,
        sign_value=1,
    )

    family = parse_source_family(pkl_path.name)
    noise = parse_noise_level(pkl_path.name)
    network = parse_network_number(pkl_path.name)

    print(f"   -> wrote {out_csv.name}  rows={len(df_out)}")

    summary.append({
        "input_pkl": pkl_path.name,
        "output_csv": out_csv.name,
        "source_family": family,
        "noise": noise,
        "network": network,
        "n_rows": len(df_out),
    })

summary_df = pd.DataFrame(summary).sort_values(["source_family", "noise", "network"])
display(summary_df)

# ----------------------------
# Create separate zip files
# ----------------------------
for family in ["GeneNetWeaver", "GeneSPIDER"]:
    family_files = sorted(OUT_DIR.glob(f"{family}_*_grn.csv"))

    zip_path = OUT_DIR / f"{family}_GRNbenchmark_submission.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for csv_path in family_files:
            zf.write(csv_path, arcname=csv_path.name)

    print(f"[SAVED] {zip_path}")
    print(f"[INFO] {family}: included {len(family_files)} CSV files")

    if len(family_files) != 15:
        print(f"[WARN] {family} does not have 15 files. Found {len(family_files)}.")

[INFO] Found 30 Piglasso result files.

[1/30] Processing: GeneNetWeaver_HighNoise_Network1_GeneExpression__Q200__bperc0.65__lam0.05-0.3x20__seed42__piglasso_results.pkl
   -> wrote GeneNetWeaver_HighNoise_Network1_grn.csv  rows=9748

[2/30] Processing: GeneNetWeaver_HighNoise_Network2_GeneExpression__Q200__bperc0.65__lam0.05-0.3x20__seed42__piglasso_results.pkl
   -> wrote GeneNetWeaver_HighNoise_Network2_grn.csv  rows=9840

[3/30] Processing: GeneNetWeaver_HighNoise_Network3_GeneExpression__Q200__bperc0.65__lam0.05-0.3x20__seed42__piglasso_results.pkl
   -> wrote GeneNetWeaver_HighNoise_Network3_grn.csv  rows=9892

[4/30] Processing: GeneNetWeaver_HighNoise_Network4_GeneExpression__Q200__bperc0.65__lam0.05-0.3x20__seed42__piglasso_results.pkl
   -> wrote GeneNetWeaver_HighNoise_Network4_grn.csv  rows=6744

[5/30] Processing: GeneNetWeaver_HighNoise_Network5_GeneExpression__Q200__bperc0.65__lam0.05-0.3x20__seed42__piglasso_results.pkl
   -> wrote GeneNetWeaver_HighNoise_Network5_grn.c

,input_pkl,output_csv,source_family,noise,network,n_rows
0,GeneNetWeaver_HighNoise_Network1_GeneExpressio...,GeneNetWeaver_HighNoise_Network1_grn.csv,GeneNetWeaver,HighNoise,Network1,9748
1,GeneNetWeaver_HighNoise_Network2_GeneExpressio...,GeneNetWeaver_HighNoise_Network2_grn.csv,GeneNetWeaver,HighNoise,Network2,9840
2,GeneNetWeaver_HighNoise_Network3_GeneExpressio...,GeneNetWeaver_HighNoise_Network3_grn.csv,GeneNetWeaver,HighNoise,Network3,9892
3,GeneNetWeaver_HighNoise_Network4_GeneExpressio...,GeneNetWeaver_HighNoise_Network4_grn.csv,GeneNetWeaver,HighNoise,Network4,6744
4,GeneNetWeaver_HighNoise_Network5_GeneExpressio...,GeneNetWeaver_HighNoise_Network5_grn.csv,GeneNetWeaver,HighNoise,Network5,9890
5,GeneNetWeaver_LowNoise_Network1_GeneExpression...,GeneNetWeaver_LowNoise_Network1_grn.csv,GeneNetWeaver,LowNoise,Network1,1234
6,GeneNetWeaver_LowNoise_Network2_GeneExpression...,GeneNetWeaver_LowNoise_Network2_grn.csv,GeneNetWeaver,LowNoise,Network2,744
7,GeneNetWeaver_LowNoise_Network3_GeneExpression...,GeneNetWeaver_LowNoise_Network3_grn.csv,GeneNetWeaver,LowNoise,Network3,864
8,GeneNetWeaver_LowNoise_Network4_GeneExpression...,GeneNetWeaver_LowNoise_Network4_grn.csv,GeneNetWeaver,LowNoise,Network4,1034
9,GeneNetWeaver_LowNoise_Network5_GeneExpression...,GeneNetWeaver_LowNoise_Network5_grn.csv,GeneNetWeaver,LowNoise,Network5,1194


[SAVED] /Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/benchmarking/results/piglasso/GRN/reformatted/GeneNetWeaver_GRNbenchmark_submission.zip
[INFO] GeneNetWeaver: included 15 CSV files
[SAVED] /Users/zoeazra/Documents/CLS/Y2/Thesis/Implementation/BurnInjuries/benchmarking/results/piglasso/GRN/reformatted/GeneSPIDER_GRNbenchmark_submission.zip
[INFO] GeneSPIDER: included 15 CSV files
